# Exercise 05 — Abstract Classes & Design Patterns

## Concepts

### Abstract Base Class (ABC)

An abstract class **cannot** be instantiated. It defines a contract that subclasses **must** implement.

```python
from abc import ABC, abstractmethod

class PaymentProcessor(ABC):
    @abstractmethod
    def process(self, amount: float) -> bool:
        """Subclasses MUST implement this."""
        pass

# PaymentProcessor()  ← TypeError! Can't instantiate abstract class

class StripeProcessor(PaymentProcessor):
    def process(self, amount):
        print(f"Charging ${amount} via Stripe")
        return True

sp = StripeProcessor()  # OK — all abstract methods implemented
```

### Common Design Patterns

| Pattern | Purpose | Real-world example |
|---------|---------|-------------------|
| **Strategy** | Swap algorithms at runtime | Sorting: quick-sort vs merge-sort |
| **Observer** | Notify multiple listeners of changes | Event systems, pub/sub |
| **Singleton** | Only one instance ever exists | Database connection pool |

---
## Task 1: Notification System (Strategy Pattern)

Build a notification system where different channels (Email, SMS, Push) can be swapped in.

```python
class NotificationChannel(ABC):
    @abstractmethod
    def send(self, recipient: str, message: str) -> str: ...

class EmailChannel(NotificationChannel): ...
class SMSChannel(NotificationChannel): ...
class PushChannel(NotificationChannel): ...

class NotificationService:
    """Uses any channel — doesn't care which one (Strategy Pattern)."""
    def __init__(self, channel: NotificationChannel): ...
    def notify(self, recipient: str, message: str) -> str: ...
```

**Rules:**
1. Each channel's `send()` returns a string: `"Email sent to {recipient}: {message}"`
2. `NotificationService.notify()` delegates to its channel's `send()`
3. `NotificationChannel` itself cannot be instantiated (it's abstract)

In [ ]:
from abc import ABC, abstractmethod

# YOUR CODE HERE

class NotificationChannel(ABC):
    pass

class EmailChannel(NotificationChannel):
    pass

class SMSChannel(NotificationChannel):
    pass

class PushChannel(NotificationChannel):
    pass

class NotificationService:
    pass

In [ ]:
# TEST — run this cell to check your solution

# Abstract class cannot be instantiated
try:
    NotificationChannel()
    assert False, "Should raise TypeError"
except TypeError:
    pass

# Email
email_svc = NotificationService(EmailChannel())
result = email_svc.notify("alice@test.com", "Hello")
assert result == "Email sent to alice@test.com: Hello"

# SMS
sms_svc = NotificationService(SMSChannel())
result = sms_svc.notify("+1234567890", "Hi there")
assert result == "SMS sent to +1234567890: Hi there"

# Push
push_svc = NotificationService(PushChannel())
result = push_svc.notify("user_42", "New update!")
assert result == "Push sent to user_42: New update!"

# Strategy swap at runtime
service = NotificationService(EmailChannel())
assert "Email" in service.notify("x", "y")
service.channel = SMSChannel()  # swap strategy
assert "SMS" in service.notify("x", "y")

print("✓ All tests passed!")

---
## Task 2: Event System (Observer Pattern)

Build a simple event emitter where listeners subscribe to events.

```python
bus = EventBus()

log = []  # we'll collect events here
bus.subscribe("user_login", lambda data: log.append(f"Login: {data}"))
bus.subscribe("user_login", lambda data: log.append(f"Audit: {data}"))

bus.emit("user_login", "alice")  # both listeners fire
# log == ["Login: alice", "Audit: alice"]
```

**Rules:**
1. `subscribe(event_name, callback)` — registers a listener
2. `emit(event_name, data)` — calls all listeners for that event
3. `unsubscribe(event_name, callback)` — removes a specific listener
4. Emitting an event with no subscribers should do nothing (no error)

In [ ]:
# YOUR CODE HERE

class EventBus:
    pass  # Replace with your implementation

In [ ]:
# TEST — run this cell to check your solution

bus = EventBus()
log = []

def on_login(data):
    log.append(f"Login: {data}")

def on_audit(data):
    log.append(f"Audit: {data}")

bus.subscribe("user_login", on_login)
bus.subscribe("user_login", on_audit)

bus.emit("user_login", "alice")
assert log == ["Login: alice", "Audit: alice"]

# Unsubscribe one listener
bus.unsubscribe("user_login", on_audit)
bus.emit("user_login", "bob")
assert log == ["Login: alice", "Audit: alice", "Login: bob"]

# Emit unknown event — no error
bus.emit("unknown_event", "data")

# Multiple event types
logout_log = []
bus.subscribe("user_logout", lambda d: logout_log.append(d))
bus.emit("user_logout", "charlie")
assert logout_log == ["charlie"]

print("✓ All tests passed!")

---
## Bonus: Singleton Pattern

A Singleton ensures only **one instance** of a class ever exists.

```python
db1 = Database.get_instance()
db2 = Database.get_instance()
assert db1 is db2  # same object!
```

Implement `Database` as a Singleton using `__new__` or a class method.

In [ ]:
# BONUS — YOUR CODE HERE

class Database:
    _instance = None
    # Hint: override __new__ or use a @classmethod get_instance()
    pass

In [ ]:
# BONUS TEST

db1 = Database()
db2 = Database()
assert db1 is db2, "Should be the exact same object"
print("✓ Singleton test passed!")